# Задание 1 (2 балла)


Выберите 5 языков в википедии (не тех, что использовались в семинаре). Скачайте по 10 случайных статей для каждого языка. Предобработайте тексты, удаляя лишние теги/отступы/разделители (если они есть). Разделите тексты на предложения и создайте датасет, в котором каждому предложению соответствует язык. Кластеризуйте тексты, используя эбмединг модель из прошлого семинара и любой алгоритм кластеризации. Проверьте качество кластеризации с помощь метрики ARI. Отдельно проанализируйте 3 ошибочно кластеризованных текста (если такие есть).

In [3]:
%pip install wikipedia

  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11756 sha256=ed5028d08a04968b582a00bd85a5e7df62c3c15a3380076838fa0d45922c6ba0
  Stored in directory: /root/.cache/pip/wheels/79/1d/c8/b64e19423cc5a2a339450ea5d145e7c8eb3d4aa2b150cde33b
Successfully built wikipedia


In [4]:
import os
import time
import wikipedia
import numpy as np

wikipedia.set_user_agent('CLHomework/1.0 (nadzeyafadzeyeva@gmail.com)')

In [5]:
import requests

MIN_INTERVAL = 0.35
_last_call = [0.0]


def _wiki_request(params):
    params['format'] = 'json'
    params.setdefault('action', 'query')
    for attempt in range(5):
        wait = MIN_INTERVAL - (time.time() - _last_call[0])
        if wait > 0:
            time.sleep(wait)
        _last_call[0] = time.time()
        r = requests.get(wikipedia.wikipedia.API_URL, params=params,
                         headers={'User-Agent': wikipedia.wikipedia.USER_AGENT}, timeout=30)
        if r.status_code == 429:
            retry_after = int(r.headers.get('Retry-After', 30))
            print(f'429 Too Many Requests, ждем {retry_after} с')
            time.sleep(retry_after)
            continue
        r.raise_for_status()
        return r.json()
    r.raise_for_status()


wikipedia.wikipedia._wiki_request = _wiki_request

In [6]:
code2lang = wikipedia.languages()
langs = ['be-tarask', 'es', 'de', 'fr', 'ja', 'kk', 'pt', 'sv', 'no', 'pl']
[(lang, code2lang[lang]) for lang  in langs]

[('be-tarask', 'беларуская (тарашкевіца)'),
 ('es', 'español'),
 ('de', 'Deutsch'),
 ('fr', 'français'),
 ('ja', '日本語'),
 ('kk', 'қазақша'),
 ('pt', 'português'),
 ('sv', 'svenska'),
 ('no', 'norsk'),
 ('pl', 'polski')]

In [ ]:
def load_with_disambigution(page):
    try:
        p = wikipedia.page(page)
    except wikipedia.DisambiguationError as e:
        random_option = np.random.choice(e.options)
        p = wikipedia.page(random_option)
    return p

def get_texts_for_lang(lang, n=10):
    wikipedia.set_lang(lang)
    wiki_content = []

    pages = wikipedia.random(n)

    while len(wiki_content) < n:
        try:
          random_page = wikipedia.random(1)
          page_name = random_page[0] if isinstance(random_page, list) else random_page
          page = load_with_disambigution(page_name)

        except Exception as e:
            print('Skipping page {}'.format(page_name), str(e).strip('\n'))
            continue

        wiki_content.append(f'{page.title}\n{page.content.replace("==", "")}')

    return wiki_content

In [ ]:
%%time
import json

WIKI_CACHE = 'wiki_texts.json'

if os.path.exists(WIKI_CACHE):
    wiki_texts = json.load(open(WIKI_CACHE))
else:
    wiki_texts = {}

wiki_texts = {}

for lang in langs:
    if wiki_texts.get(lang):
        continue
    try:
        wiki_texts[lang] = get_texts_for_lang(lang, 10)
    except Exception as e:
        print('ERROR ON - ', lang, str(e).strip('\n'))
        continue

    print(lang, len(wiki_texts[lang]))
    time.sleep(1)

json.dump(wiki_texts, open(WIKI_CACHE, 'w'), ensure_ascii=False)

be-tarask 10
es 10
Skipping page Tim McGraw Page id "tim mcgrawe" does not match any pages. Try another id!
de 10
fr 10


/usr/local/lib/python3.13/dist-packages/wikipedia/wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file /usr/local/lib/python3.13/dist-packages/wikipedia/wikipedia.py. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')


ja 10
kk 10
pt 10
Skipping page Santa Maria della Concezione "Santa Maria della Concezione" may refer to: 
Santa Maria della Concezione dei Cappuccini
Santa Maria della Concezione in Campo Marzio
Santa Maria della Concezione alla Lungara
Santa Maria della Concezione dei Sacconi Turchini
Santa Maria della Concezione delle Viperesche
Santa Maria della Concezione a Montecalvario
sv 10
Skipping page 3. århundre f.Kr. Page id "3 århundrede f.kr" does not match any pages. Try another id!
no 10
pl 10
CPU times: user 4.8 s, sys: 370 ms, total: 5.17 s
Wall time: 2min 57s


In [ ]:
import re
import pandas as pd
import nltk

nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

def preprocess_text(text):
  text = re.sub(r'==+.*?==+', ' ', text)
  text = re.sub(r'\[.*?\]', ' ', text)
  text = re.sub(r'[^\w\s.,!?\-]', ' ', text)
  text = re.sub(r'\s+', ' ', text)
  return text.strip()

sentences_list = []
languages_list = []

for lang, articles in wiki_texts.items():
  for article in articles:
    cleaned_article = preprocess_text(article)

    sentences = sent_tokenize(cleaned_article)

    for sentence in sentences:
      sentence = sentence.strip()

      if len(sentence) > 2:
        sentences_list.append(sentence)
        languages_list.append(lang)

dataset = pd.DataFrame({
    'sentence': sentences_list,
    'language': languages_list
})

dataset.head()

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


,sentence,language
0,"Melbet Melbet - кампанія, якая займаецца азарт...",be-tarask
1,Была заснавана ў 2012 годзе і зарэгістравана н...,be-tarask
2,Melbet належыць кіпрскай фірме Tutkiа.,be-tarask
3,На сайце даступна 44 моўныя версіі.,be-tarask
4,Букмекер падтрымлівае ўсе папулярныя сусветныя...,be-tarask


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

embedding_model = SentenceTransformer("intfloat/multilingual-e5-base")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
texts = dataset['sentence'].tolist()
processed_texts = [f"query: {text}" for text in texts]
X = embedding_model.encode(processed_texts, show_progress_bar=True)
cluster = KMeans(n_clusters=10, init='kmeans++', n_init=10, random_state=42)
cluster.fit(X)
labels = np.array(cluster.labels_) + 1
dataset['cluster'] = labels

dataset.head()

Batches:   0%|          | 0/54 [00:00<?, ?it/s]

,sentence,language,cluster
0,"Melbet Melbet - кампанія, якая займаецца азарт...",be-tarask,3
1,Была заснавана ў 2012 годзе і зарэгістравана н...,be-tarask,3
2,Melbet належыць кіпрскай фірме Tutkiа.,be-tarask,3
3,На сайце даступна 44 моўныя версіі.,be-tarask,3
4,Букмекер падтрымлівае ўсе папулярныя сусветныя...,be-tarask,3


In [ ]:
ari_score = adjusted_rand_score(dataset['language'], dataset['cluster'])
print("ARI Score:", ari_score)

ARI Score: 0.6027029048117544


In [ ]:
pd.crosstab(dataset['language'], dataset['cluster'])

cluster,1,2,3,4,5,6,7,8,9,10
language,,,,,,,,,,
be-tarask,0,8,92,0,2,0,0,4,0,0
de,0,43,3,0,0,124,0,47,184,1
es,228,50,0,0,0,1,0,5,0,0
fr,1,12,0,0,0,0,0,0,0,153
ja,0,14,0,0,32,0,0,9,0,0
kk,0,16,106,0,0,1,0,5,0,0
no,0,5,2,0,118,3,0,2,0,0
pl,0,18,1,142,0,0,0,4,1,0
pt,0,7,0,0,0,0,207,14,0,2


In [ ]:
print(dataset[(dataset['language'] == 'ja') & (dataset['cluster'] == 2)]['sentence'].sample(5).values)

['New York Rizzoli International Publications.'
 'Encyclopædia Britannica 英語 .' 'KARYA KONVEX INDONESIA インドネシア PT.'
 'マレーシア INTERPRINT Polska Sp.' 'Ltd. 旧 Toppan Security Printing Pte.']


In [ ]:
print(dataset[(dataset['language'] == 'de') & (dataset['cluster'] == 2)]['sentence'].sample(5).values)

['Literatur Álvaro Menanteau Historia del jazz en Chile , Ocho Libros Editores, 2003, ISBN 9789568018139, S. 56 57, 98.'
 'Raid Liiver verfasste den Text des Liedes.' 'Die Tournee endete am 27.'
 'Longmans, Green Co., London 1875 Digitalisat, englisch .'
 'Dessen erster Auftritt fand am 30.']


In [ ]:
print(dataset[(dataset['language'] == 'es') & (dataset['cluster'] == 2)]['sentence'].sample(5).values)

['Edie McClurg como Mrs. Butterworth George Johnsen as Kaptain Krispy.'
 'Foodfight Foodfight!' 'Invitados Carlos Hipólito como Juan Garayoa.'
 'Charlie Sheen como Dex Dogtective.' 'Will Keen como Albert James.']


В ходе работы было обнаружено, что второй кластер выполняет роль сборного. К нему не относится на 100% ни один из целевых языков. Я решила посмотреть, почему так, и пришла к следующим выводам:

1.   Японский: Предложения извлечены из японской Википедии, но фактически написаны на английском языке. Так как английский язык мы не брали, модель забросила их в, по ее мнению, английский кластер.
2.   Немецкий: Здесь в принципе та же ситуация, строки содержат списки литературы, коды ISBN, названия иностранных издательств и имена собственные, из-за чего мультиязычная модель сближает их с англоязычным шумом.
1.   Испанский: Сюда попали предложения, имеющие строго повторяющийся синтаксический шаблон перечисления ролей в фильмах: "Имя Актрисы como Имя Роли". Кроме того, "Имя Роли" являются именами, написанными на английском языке.

В итоге второй кластер сработал как фильтр для нестандартных текстов. Модель разложила предложения по их реальному смыслу и форме. Она смогла сама увидеть скрытый английский язык, технические данные книг и одинаковые шаблоны в разных статьях и убрала весь этот шум в одну общую группу.









## Задание 2 (2 балла)
Возьмите какой-нибудь из файлов из Syntagrus или Taiga -
https://github.com/UniversalDependencies/UD_Russian-SynTagRus/tree/master   
https://tatianashavrina.github.io/taiga_site/downloads  

Используя Dependency Matcher напишите 2 разных query к этому корпусу, результаты которых выдают что-то осмысленное (показывают какую-то закономерность; подвсечивают примеры какого-то феномена; выдают то, что можно использовать в другой задаче).
Добавьте дополнительные правила фильтрации, если получается слишком зашумленный или наоборот пустой результат (см. семинар и примеры для SVO)
Используйте именно синтактическую структуру, а не просто слова и POS тэги.
Обязательно покажите примеры, которые находятся по вашим запросам.


Ответ должен быть структурирован как-то так:
```
## query
Краткое описание того, что вы пытаетесь найти

### код
код с запросом (или несколько ячеек где вы последовательно улучшаете запрос)

### примеры
несколько самых показательных примеров или даже какая-то агрегация вроде топ-N

... (и так два раза)
```



In [ ]:
import urllib.request
import re
import spacy
from spacy.matcher import DependencyMatcher

url = "https://github.com/UniversalDependencies/UD_Russian-SynTagRus/raw/refs/heads/master/ru_syntagrus-ud-dev.conllu"

req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
with urllib.request.urlopen(req) as response:
    raw_data = response.read().decode('utf-8')

syntagrus_sentences = re.findall(r'# text = (.*)', raw_data)

raw_text = " ".join(syntagrus_sentences[:100])

nlp = spacy.load("ru_core_news_md")
doc = nlp(raw_text)

RELATIVE_PRONOUNNS = {'который', 'что'}

def resolve_relative(tok, verb):
    if tok.lemma_ in RELATIVE_PRONOUNNS and verb.dep_ == 'acl:relcl':
        return verb.head
    return tok

def phrase(tok):
    skip = [l.i for l in tok.children if l.dep_ == 'acl:relcl' for l in l.subtree]
    return ' '.join([t.text for t in tok.subtree if t.i not in skip and t.dep_ != 'punct'])

Поиск глагольных конструкций с раскрытыми синтаксическими обстоятельствами (Глагол + Обстоятельство времени/места).

In [ ]:
matcher_1 = DependencyMatcher(nlp.vocab)
pattern_1 = [
    {"RIGHT_ID": "verb", "RIGHT_ATTRS": {"POS": "VERB"}},
    {
        "LEFT_ID": "verb",
        "REL_OP": ">",
        "RIGHT_ID": "obl_noun",
        "RIGHT_ATTRS": {"DEP": "obl", "POS": "NOUN"}
    }
]
matcher_1.add("VERB_OBL", [pattern_1])

print("Найденные конструкции «Глагол + Обстоятельство»:")
results_1 = []
for _, token_ids in matcher_1(doc):
    verb = doc[token_ids[0]]
    obl = doc[token_ids[1]]
    obl_resolved = resolve_relative(obl, verb)
    results_1.append(f"{verb.lemma_} + {phrase(obl_resolved).lower()}")

for res in list(set(results_1))[:5]:
    print(res)

Найденные конструкции «Глагол + Обстоятельство»:
войти + во многие европейские языки
применяться + к способам выполнения действий с бесконечно малыми величинами
содержать + в явной или неявной форме
объясняться + в xviii веке
проникнуть + в первой половине xii века


Поиск сложных именных групп со связью «Прилагательное + Существительное».

In [ ]:
matcher_2 = DependencyMatcher(nlp.vocab)
pattern_2 = [
    {"RIGHT_ID": "noun", "RIGHT_ATTRS": {"POS": "NOUN"}},
    {
        "LEFT_ID": "noun",
        "REL_OP": ">",
        "RIGHT_ID": "amod_adj",
        "RIGHT_ATTRS": {"DEP": "amod", "POS": "ADJ"}
    }
]
matcher_2.add("NOUN_AMOD", [pattern_2])

print("Полные именные группы с их главным признаком:")
results_2 = []
for _, token_ids in matcher_2(doc):
    noun = doc[token_ids[0]]
    adj = doc[token_ids[1]]
    results_2.append(f"Полная группа: {phrase(noun).lower()} | Признак: {adj.lemma_}")

for res in list(set(results_2))[:5]:
    print(res)

Полные именные группы с их главным признаком:
Полная группа: в современной трактовке у разных реализаций одного и того же алгоритма | Признак: современный
Полная группа: как счётная мудрость | Признак: счётный
Полная группа: вычислительный процесс идущий от варьируемых исходных данных к искомому результату | Признак: вычислительный
Полная группа: венского астронома и математика георга пурбаха georg peurbach | Признак: венский
Полная группа: других трудов посвящённых всё тому же вопросу - обучению искусству счёта с помощью цифр | Признак: других


## Задание 3 (4 балла)

Эссе

Прочитайте статью *FineWeb2: One Pipeline to Scale Them All — Adapting Pre-Training Data Processing to Every Language*.
Ссылка - https://arxiv.org/pdf/2506.20920

Напишите эссе, в котором вы, используя эту статью как конкретный пример, рассуждаете о том, что значит создать «хороший» многоязычный датасет для предобучения?

Не пересказывайте статью! Выберите что-то, что вам показалось интересным (assumption, результат эксперимента, метрику и т.п.), и постройте вокруг этого аргумент. Вы можете соглашаться с авторами, спорить с ними или добавлять что-то от себя.

Можете опираться на следующие вопросы, но можете и придумать что-то свое:

- Действительно ли возможен "one pipeline to scale them all", или некоторые аспекты отбора и очистки данных по своей природе должны быть специфичны для конкретного языка?
- Что именно означает "качество данных" в FineWeb2? Согласны ли в с этим?
- Если бы вам нужно было построить корпус для предобучения на языке, который не исследовался в статье, насколько вы бы доверяли предложенному пайплайну и где ожидали бы проблемы?
- Какое решение в дизайне FineWeb2 кажется вам наиболее убедительным — или, наоборот, наименее убедительным — и почему?
- FineWeb2 охватывает более тысячи языков, однако для многих низкоресурсных языков доступно очень мало данных, причём эти данные могут быть весьма однородными. Честно ли в этом случае заявлять об их покрытии? Как провести границу?
- В статье сначала удаляются дубликаты, а затем часть повторяющихся данных выборочно возвращается с помощью *rehydration*. То есть дубли это не всегда "плохие данные". Как тогда понять, что дедублицировать, а что нет?
- Авторы тщательно выбирают *early-signal* задачи для оценки, чтобы сравнивать разные варианты датасета. Достаточно ли качества модели на downstream-задачах для оценки качества корпуса?

Вы можете предлагать гипотетические примеры, альтернативные решения или эксперименты, с помощью которых можно было бы проверить ваш аргумент.

Цель задания — не покритиковать статью. Цель — построить связный аргумент, который нельзя было бы написать, не прочитав статью внимательно и не обдумав её содержание.



FineWeb2: One Pipeline to Scale Them All — Adapting Pre-Training Data Processing to Every Language

Данная статья оставила у меня двоякое впечатление. Безусловно, авторы двигаются в правильном направлении, и предложенный ими подход действительно необходим. Пайплайн, который настраивает фильтры под статистику конкретного языка, кажется гораздо более логичным, чем прежние универсальные алгоритмы. И поскольку меня интересуют лингвистические задачи, связанные с белорусским языком, во время прочтения я постоянно соотносила предложенный метод с его особенностями. И все-таки у меня появились сомнения. По моему мнению, алгоритм, созданный сразу для тысячи языков, физически не способен качественно охватить и учесть все тонкости отдельно взятой языковой среды.

В вопросах чисто технической фильтрации предложенному пайплайну действительно можно доверять. Авторы создали надежный инструмент для удаления рекламного спама, дубликатов страниц и поврежденного кода, все это работает эффективно. Их математические решения, такие как метод нечеткой дедупликации MinHash или алгоритм «регидратации» текста, универсальны и не зависят от конкретной языковой группы. Использовать FineWeb2 в качестве инструмента первичной фильтрации это вполне разумное решение.

Однако сомнения возникают, когда мы переходим к анализу самого языка. Первая серьезная проблема для белорусского контекста – это этап идентификации языка (LID). Авторы честно пишут в статье, что, если у исследуемого языка есть близкородственный высокоресурсный сосед, классификатор GlotLID начинает ошибаться из-за сильного дисбаланса данных в интернете. Для белорусского языка таким соседом является русский. Белорусское интернет-пространство специфично: там много текстов с элементами двуязычия или использованием «трасянки». Поскольку алгоритм FineWeb2 ищет чистый язык и выставляет пороги на основе медианы распределения, будет либо отсечена огромная часть реальных белорусских текстов, принятых за испорченный русский язык, либо допущено массовое загрязнение корпуса русскоязычным контентом. Предложенные списки уникальных слов-маркеров здесь мало чем помогут, так как лексическое сходство языков слишком велико.

Второй повод для сомнений – это риск получить крайне однообразный корпус из-за использования Википедии как главного эталона качества. Авторы сами обращают внимание на проблему разнообразия: после работы фильтров корпуса для малопредставленных языков более чем наполовину состоят из текстов Библии и Википедии. Алгоритм берёт Википедию за образец правильного языка и по её стандартам оценивает все остальные сайты в интернете. Но белорусская Википедия сама по себе довольно мала, а язык в ней сухой и строго академический. Если алгоритм FineWeb2 настроит свои фильтры качества, ориентируясь только на этот узкий эталон, он просто не пропустит большинство обычных сайтов. Любые современные новостные статьи, посты в блогах, обсуждения в социальных сетях или живая художественная литература будут казаться алгоритму «низкокачественным контентом» просто потому, что они не похожи на структуру Википедии. Фильтры Gopher Quality и FineWeb Quality сочтут разговорные обороты, неформальные фразы, слова из «трасянки» и новые заимствованные слова мусором и просто удалят их, лишив корпус естественного языкового разнообразия.

Более того, в белорусской цифровой среде существуют две Википедии: одна на официальном правописании, вторая – на тарашкевице. Они используют разные орфографические стандарты и разные наборы стоп-слов. Попытка алгоритма FineWeb2 автоматически усреднить статистику по двум разным стандартам приведет к тому, что система не сможет определить единую норму языка, выставит некорректные пороги и одинаково плохо очистит оба текста.

Кроме того, у меня есть вопросы к выбору токенизатора. Авторы взяли за основу токенизатор от модели Gemma из-за его большого словаря и хорошей компактности на тестовых языках. Но они тестировали его на кириллице только на примере русского языка. Белорусский язык имеет свои уникальные графемы, например, букву «ў» или апостроф. Если токенизатор Gemma не имеет этих символов в своем базовом словаре, он начнет дробить обычные белорусские слова буквально по буквам. Это искусственно завысит показатель избыточности субслов, и в результате фильтры FineWeb2 могут ошибочно принять нормальные белорусские слова за бессмысленную белиберду и удалить их из корпуса.

Подводя итоги, как уже упоминалось, пайплайн эффективен как инструмент для первого, самого чернового этапа работы. Но, как справедливо резюмируют сами авторы, пайплайны, разработанные вручную экспертами по конкретному языку, всё еще превосходят автоматический подход. Для построения качественного белорусского корпуса нам неизбежно придется корректировать этот алгоритм вручную: отдельно защищать тексты от влияния рунета и искусственно смягчать фильтры качества, чтобы не уничтожить живую специфику «магутнай беларускай мовы»!